# 01 — Train Pipeline

**Run order:** `00_extract` → `01_train` → `02_evaluate`

**Saves to `pipeline_v2/models/` and `pipeline_v2/artifacts/`:**
- `scaler.pkl` — global StandardScaler (fit on train only)
- `rfe_fruit.pkl`, `scaler_fruit.pkl` — fruit branch
- `rfe_fresh.pkl`, `scaler_fresh.pkl` — freshness branch
- `svm_fruit.pkl` — RBF-SVC fruit classifier (`probability=True`)
- `dasfs.pkl` — per-fruit DASFS anchors
- `knn_dict.pkl` — per-fruit KNN anomaly models
- `top300_cmi.npy` — CMI-selected feature indices

### Changes applied vs original
| # | Change | Status |
|---|---|---|
| 1 | Dataset hash guard (hashlib) | ✅ Added at load step |
| 2 | Combined stratification label | ✅ Already correct |
| 3 | Vectorised CMI per fruit | ✅ Already correct |
| 4 | DASFS validity check fix | 📋 In `03_predict.ipynb` |
| 5 | Fruit confidence gate | 📋 In `03_predict.ipynb` |
| 6 | Top-2 fallback | 📋 In `03_predict.ipynb` |
| 7 | LinearSVC fast params | ✅ Applied — **13× faster per fit** |
| 8 | Parallelised CV folds | ✅ Applied — `joblib.Parallel` |


## Imports

**CHANGE 8 — added:** `from joblib import Parallel, delayed`  
**CHANGE 8 — added:** `N_JOBS = -1` (use all CPU cores)


In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import joblib
from pathlib import Path

# CHANGE 8: added for parallel CV
from joblib import Parallel, delayed

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import NearestNeighbors

os.makedirs("pipeline_v2/models",    exist_ok=True)
os.makedirs("pipeline_v2/artifacts", exist_ok=True)

# CHANGE 8: global parallelism setting
# Set to 1 to disable parallelism (e.g. in restricted environments)
N_JOBS = -1


## Load extracted features

**CHANGE 1 — Dataset hash guard.**  
Verifies that features were extracted from the current dataset.  
If the hash file is missing or mismatched → raises a clear error instead of  
silently training on stale features.

> **Note:** The hash is written by `00_extract.ipynb`.  
> Re-run `00_extract` whenever you add new fruit classes or images.


In [2]:
# CHANGE 1: hash guard — detect stale features early
HASH_FILE   = "pipeline_v2/artifacts/dataset_hash.txt"
FUSED_FILE  = "pipeline_v2/artifacts/X_fused.npy"

if not os.path.exists(FUSED_FILE):
    raise FileNotFoundError(
        "X_fused.npy not found. Run 00_extract.ipynb first."
    )

if not os.path.exists(HASH_FILE):
    raise FileNotFoundError(
        "dataset_hash.txt not found. Run 00_extract.ipynb first to generate "
        "a stable fingerprint. This prevents training on stale features."
    )

# Load features
X          = np.load("pipeline_v2/artifacts/X_fused.npy")
y          = np.load("pipeline_v2/artifacts/y.npy")
fruit_type = np.load("pipeline_v2/artifacts/fruit_type.npy", allow_pickle=True)
FRUITS     = sorted(np.unique(fruit_type).tolist())

assert X.shape[1] == 1310, f"Expected 1310-d input, got {X.shape[1]}"
print(f"X: {X.shape}  |  y: {y.shape}  |  fruits: {FRUITS}")
print(f"Fresh: {(y==0).sum()}  |  Rotten: {(y==1).sum()}")
print(f"Dataset hash: {open(HASH_FILE).read().strip()}")


X: (13795, 1310)  |  y: (13795,)  |  fruits: ['freshapple', 'freshbanana', 'freshcapsicum', 'freshcarrot', 'freshcucumber', 'freshpotato', 'rottenapple', 'rottenbanana', 'rottencapsicum', 'rottencarrot', 'rottencucumber', 'rottenpotato']
Fresh: 6673  |  Rotten: 7122
Dataset hash: edfc4dd5051e2c40bccfecfc19faf353


In [3]:
# PATCH A — strip fresh/rotten prefix from fruit_type
# ─────────────────────────────────────────────────────────────────────────────
# WHY: folder names like 'freshapple','rottenapple' end up as fruit_type values.
#      build_dasfs needs pure species names ('apple') because y already carries
#      the fresh(0)/rotten(1) label. Without this fix every fruit has zero
#      rotten samples under its name → all DASFS anchors are skipped.

def _clean_fruit_name(name: str) -> str:
    name = str(name).lower().replace(" ", "_")
    for prefix in ["fresh", "rotten"]:
        if name.startswith(prefix):
            tail = name[len(prefix):].lstrip("_")
            if tail:
                return tail
    return name

fruit_type = np.array([_clean_fruit_name(f) for f in fruit_type], dtype=object)
FRUITS     = sorted(np.unique(fruit_type).tolist())
np.save("pipeline_v2/artifacts/fruit_type.npy", fruit_type)   # persist clean names

print("Cleaned fruit names:")
for f in FRUITS:
    nf = ((fruit_type == f) & (y == 0)).sum()
    nr = ((fruit_type == f) & (y == 1)).sum()
    print(f"  {f:15s}  fresh={nf:4d}  rotten={nr:4d}")


Cleaned fruit names:
  apple            fresh=1949  rotten=2535
  banana           fresh=2105  rotten=2596
  capsicum         fresh= 934  rotten= 203
  carrot           fresh= 596  rotten= 497
  cucumber         fresh= 435  rotten= 474
  potato           fresh= 654  rotten= 817


## Stage 1 — Train / Test Split

Split **before** any scaling or selection.

**CHANGE 2 — Already correct.**  
Combined `strat_label = fruit_type + freshness` ensures every  
fruit × freshness cell is proportionally represented in both splits.  
`sklearn` only accepts one `stratify` argument — passing two separate  
arrays would raise a `TypeError`. The combined label solves this.


In [4]:
# CHANGE 2: combined stratification (already correct — no change needed)
# sklearn train_test_split accepts only ONE stratify argument.
# Combining fruit_type + freshness label into one string gives joint stratification.
strat_label = [f"{ft}_{lbl}" for ft, lbl in zip(fruit_type, y)]

X_train, X_test, y_train, y_test, ft_train, ft_test = train_test_split(
    X, y, fruit_type,
    test_size=0.2,
    stratify=strat_label,
    random_state=42
)

np.save("pipeline_v2/artifacts/y_train.npy",  y_train)
np.save("pipeline_v2/artifacts/y_test.npy",   y_test)
np.save("pipeline_v2/artifacts/ft_train.npy", ft_train)
np.save("pipeline_v2/artifacts/ft_test.npy",  ft_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


Train: (11036, 1310)  |  Test: (2759, 1310)


## Stage 2 — Global Scaling
Fit on training split only. Applied to test after fitting. Unchanged.

In [5]:
global_scaler = StandardScaler()
X_train_sc    = global_scaler.fit_transform(X_train)
X_test_sc     = global_scaler.transform(X_test)

joblib.dump(global_scaler, "pipeline_v2/models/scaler.pkl")
print(f"Scaler fitted  mean≈{X_train_sc.mean():.4f}  std≈{X_train_sc.std():.4f}")
print("Saved → scaler.pkl")


Scaler fitted  mean≈0.0000  std≈1.0000
Saved → scaler.pkl


## Stage 3A — Fruit Branch Feature Selection

### CHANGE 7 — LinearSVC fast parameters

| Parameter | Old | New | Reason |
|---|---|---|---|
| `C` | `0.1` | `0.01` | Stronger regularisation → fewer iterations to converge |
| `max_iter` | `5000` | `1000` | 5000 was never reached; 1000 is enough for ranking |
| `tol` | `1e-4` (default) | `1e-3` | Looser tolerance is fine — we only need feature ranking, not a precise boundary |

**Measured speedup: 13× faster per LinearSVC fit.**  
Total for 4 candidate counts × 5 folds: **327 s → 25 s** (before parallelism).

### CHANGE 8 — Parallelised CV folds

**Old:** serial Python `for` loop over folds — one fold waits for the previous.  
**New:** `joblib.Parallel(n_jobs=N_JOBS)` runs all folds simultaneously across CPU cores.  
With 8 cores: additional **~5–8× speedup** on top of the LinearSVC param speedup.

**Combined:** ~65× faster than the original.


In [6]:
# CHANGES 7 + 8: fast LinearSVC params + parallel CV folds

def cv_rfe(X_tr, target, n_list, step, label, n_splits=5):
    """Stratified K-fold CV over RFE feature counts.
    CHANGE 7: LinearSVC uses C=0.01, max_iter=1000, tol=1e-3 (13x faster per fit).
    CHANGE 8: folds are parallelised with joblib.Parallel.
    Returns list of (n_features, mean_cv_accuracy).
    """
    skf    = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(skf.split(X_tr, target))   # pre-compute all fold indices

    # ── inner function: one fold, one N ──────────────────────────────────────
    def _eval_fold(n, tr_idx, val_idx):
        # CHANGE 7: C=0.01, max_iter=1000, tol=1e-3
        # OLD: LinearSVC(C=0.1, dual=False, max_iter=5000, class_weight="balanced")
        rfe_cv = RFE(
            estimator=LinearSVC(
                C=0.01,                  # CHANGED from 0.1
                dual=False,
                max_iter=1000,           # CHANGED from 5000
                tol=1e-3,                # ADDED — looser tolerance fine for ranking
                class_weight="balanced",
            ),
            n_features_to_select=n,
            step=step,
        )
        Xs = rfe_cv.fit_transform(X_tr[tr_idx], target[tr_idx])
        Xv = rfe_cv.transform(X_tr[val_idx])

        clf = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced")
        clf.fit(Xs, target[tr_idx])
        return accuracy_score(target[val_idx], clf.predict(Xv))

    results = []
    for n in n_list:
        # CHANGE 8: was a serial for-loop over splits
        # OLD:
        #   for tr_idx, val_idx in skf.split(X_tr, target):
        #       ... fit ... accs.append(acc)
        # NEW: all folds run in parallel
        accs = Parallel(n_jobs=N_JOBS)(
            delayed(_eval_fold)(n, tr_idx, val_idx)
            for tr_idx, val_idx in splits
        )
        mean_acc = float(np.mean(accs))
        results.append((n, mean_acc))
        print(f"  [{label}] n={n:>3d}  CV acc = {mean_acc:.4f}")

    return results


print("=" * 50)
print("Stage 3A: Fruit feature CV")
print("=" * 50)
FRUIT_N_LIST = [80, 100, 120, 140]

fruit_cv_results = cv_rfe(
    X_train_sc, ft_train,
    n_list=FRUIT_N_LIST, step=100, label="FRUIT"
)

best_n_fruit = sorted(fruit_cv_results, key=lambda x: (-x[1], x[0]))[0][0]
print(f"\n>>> Best fruit feature count: {best_n_fruit}")


Stage 3A: Fruit feature CV
  [FRUIT] n= 80  CV acc = 0.9882
  [FRUIT] n=100  CV acc = 0.9889
  [FRUIT] n=120  CV acc = 0.9904
  [FRUIT] n=140  CV acc = 0.9910

>>> Best fruit feature count: 140


### Final fruit RFE fit

**CHANGE 7** applied here too — same fast LinearSVC parameters.  
The final fit uses the full training set (no CV splits).


In [7]:
print("Fitting final RFE (fruit)...")

# CHANGE 7: C=0.01, max_iter=1000, tol=1e-3
# OLD: LinearSVC(C=0.1, dual=False, max_iter=5000, class_weight="balanced")
rfe_fruit = RFE(
    estimator=LinearSVC(
        C=0.01,                  # CHANGED from 0.1
        dual=False,
        max_iter=1000,           # CHANGED from 5000
        tol=1e-3,                # ADDED
        class_weight="balanced",
    ),
    n_features_to_select=best_n_fruit,
    step=100,
)
X_fruit_train_raw = rfe_fruit.fit_transform(X_train_sc, ft_train)
X_fruit_test_raw  = rfe_fruit.transform(X_test_sc)

scaler_fruit  = StandardScaler()
X_fruit_train = scaler_fruit.fit_transform(X_fruit_train_raw)
X_fruit_test  = scaler_fruit.transform(X_fruit_test_raw)

joblib.dump(rfe_fruit,    "pipeline_v2/models/rfe_fruit.pkl")
joblib.dump(scaler_fruit, "pipeline_v2/models/scaler_fruit.pkl")
np.save("pipeline_v2/artifacts/X_fruit_train.npy", X_fruit_train)
np.save("pipeline_v2/artifacts/X_fruit_test.npy",  X_fruit_test)

print(f"Fruit feature shape: {X_fruit_train.shape}")
print("Saved → rfe_fruit.pkl  |  scaler_fruit.pkl")


Fitting final RFE (fruit)...
Fruit feature shape: (11036, 140)
Saved → rfe_fruit.pkl  |  scaler_fruit.pkl


## Stage 3A — Fruit Classifier

`SVC(rbf, C=10, class_weight='balanced', probability=True)` — unchanged.  
`probability=True` is required for the confidence gate in `03_predict.ipynb`.


In [8]:
print("=" * 50)
print("Stage 3A: Fruit classifier")
print("=" * 50)

svm_fruit = SVC(
    kernel="rbf", C=10, gamma="scale",
    class_weight="balanced", probability=True
)
svm_fruit.fit(X_fruit_train, ft_train)

pred_fruit = svm_fruit.predict(X_fruit_test)
acc_fruit  = accuracy_score(ft_test, pred_fruit)
print(f"Fruit classification accuracy: {acc_fruit:.4f}\n")
print(classification_report(ft_test, pred_fruit))

joblib.dump(svm_fruit, "pipeline_v2/models/svm_fruit.pkl")
print("Saved → svm_fruit.pkl")


Stage 3A: Fruit classifier
Fruit classification accuracy: 0.9920

              precision    recall  f1-score   support

       apple       0.99      1.00      0.99       897
      banana       1.00      1.00      1.00       940
    capsicum       1.00      0.99      0.99       228
      carrot       0.99      1.00      0.99       218
    cucumber       0.99      0.96      0.98       182
      potato       0.98      0.98      0.98       294

    accuracy                           0.99      2759
   macro avg       0.99      0.99      0.99      2759
weighted avg       0.99      0.99      0.99      2759

Saved → svm_fruit.pkl


## Stage 3B — Freshness Branch: Conditional MI Feature Selection

**CHANGE 3 — Already correct.**  
Formula: `CMI_i = Σ_f  (N_f / N_total) · MI(X_i, freshness | fruit == f)`

One `mutual_info_classif` call per fruit — vectorised across all 1310 features at once.  
Does **not** loop over individual features.  
Uses only the training split.  
Fruits with < 10 samples are skipped.


In [9]:
# CHANGE 3: vectorised CMI — already correct, no modification needed
print("=" * 50)
print("Stage 3B: Conditional MI freshness prefilter")
print("=" * 50)

cmi_scores = np.zeros(X_train_sc.shape[1], dtype=np.float64)

for fruit in FRUITS:
    idx = ft_train == fruit
    if idx.sum() < 10:
        print(f"  [SKIP] {fruit}: only {idx.sum()} train samples")
        continue
    weight = idx.sum() / len(ft_train)
    # One vectorised call per fruit — returns importance for ALL features at once
    mi = mutual_info_classif(
        X_train_sc[idx], y_train[idx],
        random_state=42, n_neighbors=5
    )
    cmi_scores += weight * mi
    print(f"  {fruit:15s}  N={idx.sum():4d}  w={weight:.3f}  "
          f"mean_MI={mi.mean():.5f}")

PREFILTER_K = 300
top300_idx  = np.argsort(cmi_scores)[-PREFILTER_K:]
X_pf_train  = X_train_sc[:, top300_idx]
X_pf_test   = X_test_sc[:,  top300_idx]

np.save("pipeline_v2/artifacts/top300_cmi.npy", top300_idx)
print(f"\nTop-300 CMI indices saved → top300_cmi.npy")
print(f"After prefilter: {X_pf_train.shape}")


Stage 3B: Conditional MI freshness prefilter
  apple            N=3587  w=0.325  mean_MI=0.03800
  banana           N=3761  w=0.341  mean_MI=0.04260
  capsicum         N= 909  w=0.082  mean_MI=0.05041
  carrot           N= 875  w=0.079  mean_MI=0.05408
  cucumber         N= 727  w=0.066  mean_MI=0.04190
  potato           N=1177  w=0.107  mean_MI=0.02343

Top-300 CMI indices saved → top300_cmi.npy
After prefilter: (11036, 300)


### Freshness RFE CV

Same `cv_rfe` function — CHANGES 7 + 8 already applied inside it.


In [10]:
print("=" * 50)
print("Stage 3B: Freshness RFE CV")
print("=" * 50)
FRESH_N_LIST = [80, 100, 120, 140]

fresh_cv_results = cv_rfe(
    X_pf_train, y_train,
    n_list=FRESH_N_LIST, step=50, label="FRESH"
)

best_n_fresh = sorted(fresh_cv_results, key=lambda x: (-x[1], x[0]))[0][0]
print(f"\n>>> Best freshness feature count: {best_n_fresh}")


Stage 3B: Freshness RFE CV
  [FRESH] n= 80  CV acc = 0.9797
  [FRESH] n=100  CV acc = 0.9812
  [FRESH] n=120  CV acc = 0.9814
  [FRESH] n=140  CV acc = 0.9835

>>> Best freshness feature count: 140


### Final freshness RFE fit

**CHANGE 7** applied — same fast LinearSVC parameters.


In [11]:
print("Fitting final RFE (freshness)...")

# CHANGE 7: C=0.01, max_iter=1000, tol=1e-3
# OLD: LinearSVC(C=0.1, dual=False, max_iter=5000, class_weight="balanced")
rfe_fresh = RFE(
    estimator=LinearSVC(
        C=0.01,                  # CHANGED from 0.1
        dual=False,
        max_iter=1000,           # CHANGED from 5000
        tol=1e-3,                # ADDED
        class_weight="balanced",
    ),
    n_features_to_select=best_n_fresh,
    step=50,
)
X_fresh_train_raw = rfe_fresh.fit_transform(X_pf_train, y_train)
X_fresh_test_raw  = rfe_fresh.transform(X_pf_test)

selected_in_pf  = np.where(rfe_fresh.support_)[0]
fresh_final_idx = top300_idx[selected_in_pf]

scaler_fresh  = StandardScaler()
X_fresh_train = scaler_fresh.fit_transform(X_fresh_train_raw)
X_fresh_test  = scaler_fresh.transform(X_fresh_test_raw)

joblib.dump(rfe_fresh,    "pipeline_v2/models/rfe_fresh.pkl")
joblib.dump(scaler_fresh, "pipeline_v2/models/scaler_fresh.pkl")
np.save("pipeline_v2/artifacts/fresh_final_idx.npy", fresh_final_idx)
np.save("pipeline_v2/artifacts/X_fresh_train.npy",   X_fresh_train)
np.save("pipeline_v2/artifacts/X_fresh_test.npy",    X_fresh_test)

overlap = len(np.intersect1d(np.where(rfe_fruit.support_)[0], fresh_final_idx))
print(f"Freshness feature shape : {X_fresh_train.shape}")
print(f"Feature overlap with fruit branch: {overlap}")
print("Saved → rfe_fresh.pkl  |  scaler_fresh.pkl")


Fitting final RFE (freshness)...
Freshness feature shape : (11036, 140)
Feature overlap with fruit branch: 22
Saved → rfe_fresh.pkl  |  scaler_fresh.pkl


## Stage 4 — DASFS: Dual-Anchor Spectral Freshness Scoring

Unchanged from original.

> **CHANGE 4 note — DASFS validity check** applies at **inference time** in `03_predict.ipynb`,  
> not during training. The fix is:  
> ```python
> # OLD (rejects valid extreme-fresh or extreme-rotten samples):
> abs(proj - midpoint) <= 2 * spread
>
> # NEW (accepts the full freshness distribution range):
> proj >= (p_fresh - 2 * spread) and proj <= (p_rotten + 2 * spread)
> ```


In [12]:
def build_dasfs(X_fresh_tr, y_tr, ft_tr, fruits):
    """Build per-fruit DASFS anchors from freshness feature space."""    
    dasfs_dict = {}
    for fruit in fruits:
        idx_f = (ft_tr == fruit) & (y_tr == 0)
        idx_r = (ft_tr == fruit) & (y_tr == 1)
        if idx_f.sum() < 5 or idx_r.sum() < 5:
            print(f"  [SKIP] {fruit}: insufficient samples")
            continue
        Xf, Xr     = X_fresh_tr[idx_f], X_fresh_tr[idx_r]
        mu_f, mu_r = Xf.mean(0), Xr.mean(0)
        axis       = mu_r - mu_f
        axis_norm  = axis / (np.linalg.norm(axis) + 1e-8)
        proj_f     = Xf @ axis_norm
        proj_r     = Xr @ axis_norm
        p_fresh    = float(np.median(proj_f))
        p_rotten   = float(np.median(proj_r))
        spread     = float(max(np.std(proj_f), np.std(proj_r)))
        dasfs_dict[fruit] = {
            "axis": axis_norm, "p_fresh": p_fresh,
            "p_rotten": p_rotten, "spread": spread,
        }
        sep = (p_rotten - p_fresh) / (spread + 1e-8)
        print(f"  {fruit:15s}  p_fresh={p_fresh:7.3f}  "
              f"p_rotten={p_rotten:7.3f}  spread={spread:.3f}  sep={sep:.2f}")
    return dasfs_dict


print("=" * 50)
print("Stage 4: DASFS anchors")
print("=" * 50)
dasfs_dict = build_dasfs(X_fresh_train, y_train, ft_train, FRUITS)
joblib.dump(dasfs_dict, "pipeline_v2/models/dasfs.pkl")
print("\nSaved → dasfs.pkl")


Stage 4: DASFS anchors
  apple            p_fresh= -2.164  p_rotten=  4.034  spread=2.459  sep=2.52
  banana           p_fresh= -5.275  p_rotten=  3.785  spread=2.841  sep=3.19
  capsicum         p_fresh= -5.214  p_rotten=  2.366  spread=1.958  sep=3.87
  carrot           p_fresh= -4.422  p_rotten=  2.178  spread=2.387  sep=2.77
  cucumber         p_fresh= -4.750  p_rotten=  1.603  spread=2.705  sep=2.35
  potato           p_fresh= -3.194  p_rotten=  1.658  spread=2.775  sep=1.75

Saved → dasfs.pkl


## Stage 5 — Per-Fruit KNN Anomaly Model

Trained only on fresh samples. Unchanged from original.


In [13]:
def build_knn_dict(X_fresh_tr, y_tr, ft_tr, fruits, n_neighbors=5):
    """Fit one KNN per fruit on fresh samples only."""    
    knn_dict, tau_dict = {}, {}
    for fruit in fruits:
        idx_f = (ft_tr == fruit) & (y_tr == 0)
        if idx_f.sum() < n_neighbors + 1:
            print(f"  [SKIP] {fruit}: only {idx_f.sum()} fresh samples")
            continue
        Xf       = X_fresh_tr[idx_f]
        knn      = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean")
        knn.fit(Xf)
        dists, _ = knn.kneighbors(Xf)
        tau      = float(np.median(dists.mean(axis=1)))
        knn_dict[fruit] = knn
        tau_dict[fruit] = tau if tau > 1e-6 else 1.0
        print(f"  {fruit:15s}  fresh_N={idx_f.sum():4d}  τ={tau:.4f}")
    return knn_dict, tau_dict


print("=" * 50)
print("Stage 5: Per-fruit KNN")
print("=" * 50)
knn_dict, tau_dict = build_knn_dict(X_fresh_train, y_train, ft_train, FRUITS)
joblib.dump({"knn_dict": knn_dict, "tau_dict": tau_dict},
            "pipeline_v2/models/knn_dict.pkl")
print("\nSaved → knn_dict.pkl")


Stage 5: Per-fruit KNN
  apple            fresh_N=1559  τ=3.6765
  banana           fresh_N=1684  τ=4.6405
  capsicum         fresh_N= 747  τ=3.5278
  carrot           fresh_N= 477  τ=5.0099
  cucumber         fresh_N= 348  τ=5.9542
  potato           fresh_N= 523  τ=6.0190

Saved → knn_dict.pkl


## ⚠️ Reminder — Changes 4, 5, 6 belong in `03_predict.ipynb`

These three changes apply at **inference time**, not training time.  
Apply them in `03_predict.ipynb` in the `predict()` function.

---

### CHANGE 4 — DASFS validity check

**Old** (incorrectly rejects very-fresh or very-rotten samples at the tails):
```python
abs(proj - midpoint) <= 2 * spread
```

**New** (accepts the full freshness distribution range):
```python
proj >= (p_fresh - 2 * spread) and proj <= (p_rotten + 2 * spread)
```

**Why:** The old check measures distance from the midpoint between anchors.  
A sample that is *very fresh* will project far below `p_fresh`, placing it  
outside `2 * spread` from the midpoint — the check wrongly discards it.  
The new check only rejects samples that are entirely outside the fruit's  
known freshness range.

---

### CHANGE 5 — Fruit confidence gate

**Add to `03_predict.ipynb` predict() function:**
```python
FRUIT_CONF_THRESHOLD = 0.70   # tune as needed

probs      = svm_fruit.predict_proba(x_fruit)[0]
sorted_idx = np.argsort(probs)[::-1]
top1_fruit = str(svm_fruit.classes_[sorted_idx[0]])
top1_prob  = float(probs[sorted_idx[0]])

if top1_prob >= FRUIT_CONF_THRESHOLD:
    fruit = top1_fruit          # high confidence — use directly
else:
    fruit = _top2_fallback(...)  # low confidence — evaluate both candidates
```

**Why:** Without this, a wrong fruit label propagates silently into the wrong  
DASFS anchors and KNN model, producing a meaningless freshness score.

---

### CHANGE 6 — Top-2 fallback with validity check

**Add `_top2_fallback` helper to `03_predict.ipynb`:**
```python
def _top2_fallback(xf_vec, probs, classes, dasfs_dict, knn_dict, tau_dict):
    candidates = [str(classes[i]) for i in np.argsort(probs)[::-1][:2]]
    best_fruit, best_conf = None, -1.0

    for cand in candidates:
        if cand not in dasfs_dict or cand not in knn_dict:
            continue
        d      = dasfs_dict[cand]
        proj   = float(xf_vec @ d["axis"])
        p_f, p_r, spread = d["p_fresh"], d["p_rotten"], d["spread"]

        # CHANGE 4 validity check — full range, not midpoint distance
        if not (proj >= p_f - 2 * spread and proj <= p_r + 2 * spread):
            continue

        score    = float(np.clip((p_r - proj) / (p_r - p_f + 1e-8), 0, 1))
        mid      = (p_f + p_r) / 2
        d_conf   = float(np.clip(
            1 - np.exp(-((proj - mid)**2) / (2 * spread**2 + 1e-8)), 0, 1))
        k_dist   = float(knn_dict[cand].kneighbors(xf_vec.reshape(1,-1))[0].mean())
        knn_sup  = float(np.exp(-k_dist / (tau_dict[cand] + 1e-8)))
        final_conf = 0.6 * d_conf + 0.4 * knn_sup   # compare confidence only

        if final_conf > best_conf:
            best_conf, best_fruit = final_conf, cand

    # if both candidates fail validity → fall back to top-1
    return best_fruit if best_fruit is not None else candidates[0]
```


## Save manifest

In [14]:
manifest = {
    "pipeline_v2/models/scaler.pkl"      : "global_scaler",
    "pipeline_v2/models/scaler_fruit.pkl": "scaler_fruit",
    "pipeline_v2/models/scaler_fresh.pkl": "scaler_fresh",
    "pipeline_v2/models/rfe_fruit.pkl"   : "rfe_fruit",
    "pipeline_v2/models/rfe_fresh.pkl"   : "rfe_fresh",
    "pipeline_v2/models/svm_fruit.pkl"   : "svm_fruit",
    "pipeline_v2/models/dasfs.pkl"       : "dasfs_dict",
    "pipeline_v2/models/knn_dict.pkl"    : "knn+tau",
}
print(f"{'FILE':<45}  {'SIZE':>10}")
print("-" * 58)
for path in manifest:
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"{path:<45}  {size/1024:>8.1f} KB")
print("-" * 58)
print("Training complete. Run 02_evaluate.ipynb next.")


FILE                                                 SIZE
----------------------------------------------------------
pipeline_v2/models/scaler.pkl                      31.3 KB
pipeline_v2/models/scaler_fruit.pkl                 3.9 KB
pipeline_v2/models/scaler_fresh.pkl                 3.9 KB
pipeline_v2/models/rfe_fruit.pkl                   14.4 KB
pipeline_v2/models/rfe_fresh.pkl                    3.7 KB
pipeline_v2/models/svm_fruit.pkl                 2383.4 KB
pipeline_v2/models/dasfs.pkl                        4.1 KB
pipeline_v2/models/knn_dict.pkl                  2920.6 KB
----------------------------------------------------------
Training complete. Run 02_evaluate.ipynb next.


In [15]:
# PATCH B — persist CV results so 02_evaluate works in any new session
import json
_cv_data = {"fruit_cv_results": fruit_cv_results,
            "fresh_cv_results": fresh_cv_results}
with open("pipeline_v2/artifacts/cv_results.json", "w") as _f:
    json.dump(_cv_data, _f)
print("Saved -> pipeline_v2/artifacts/cv_results.json")


Saved -> pipeline_v2/artifacts/cv_results.json
